In [1]:
%pip install -q coqui-tts coqui-tts-trainer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import pandas as pd, numpy as np, soundfile as sf, librosa
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path("/kaggle/input/datasets/alaaldeais/najdi-dataset")
OUT_DIR   = Path("/kaggle/working/xtts_data")
WAV_OUT   = OUT_DIR / "wavs"
WAV_OUT.mkdir(parents=True, exist_ok=True)
TARGET_SR, SPEAKER = 22050, "najdi_speaker"
TEST_SESSIONS, EVAL_SESSIONS = [14], [13]

df = pd.read_csv(DATA_ROOT / "metadata.csv")
on_disk = {p.name for p in (DATA_ROOT / "wavs").glob("*.wav")}
df["file_name"] = df["audio_path"].astype(str).apply(lambda p: Path(p).name)
df = df[df["file_name"].isin(on_disk)].reset_index(drop=True)
print("Matched clips:", len(df))

rows = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Converting"):
    out_name = f"clip_{int(r['id']):05d}.wav"
    out_path = WAV_OUT / out_name
    if not out_path.exists():
        wav, sr = sf.read(DATA_ROOT / "wavs" / r["file_name"], dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
        _, (s, e) = librosa.effects.trim(wav, top_db=40)
        pad = int(0.15 * TARGET_SR)
        wav = wav[max(0, s - pad):min(len(wav), e + pad)]
        sf.write(out_path, wav, TARGET_SR, subtype="PCM_16")
    rows.append({"audio_file": f"wavs/{out_name}", "text": str(r["Text"]).strip(),
                 "speaker_name": SPEAKER, "recorded_at": r["recorded_at"],
                 "dur": sf.info(out_path).duration})

meta = pd.DataFrame(rows)
meta["recorded_at"] = pd.to_datetime(meta["recorded_at"])
meta = meta.sort_values("recorded_at").reset_index(drop=True)
meta["session"] = (meta["recorded_at"].diff() > pd.Timedelta(minutes=30)).cumsum()

meta["split"] = "train"
meta.loc[meta["session"].isin(EVAL_SESSIONS), "split"] = "eval"
meta.loc[meta["session"].isin(TEST_SESSIONS), "split"] = "test"
for split in ["train", "eval", "test"]:
    part = meta[meta["split"] == split]
    part[["audio_file", "text", "speaker_name"]].to_csv(OUT_DIR / f"metadata_{split}.csv", sep="|", index=False)
    print(f"{split:5s}: {len(part):3d} clips | {part['dur'].sum()/60:5.1f} min")
meta.to_csv(OUT_DIR / "metadata_full_with_sessions.csv", index=False)
print("Data ready ✅")

Matched clips: 756


Converting:   0%|          | 0/756 [00:00<?, ?it/s]

train: 701 clips |  92.6 min
eval :  35 clips |   6.2 min
test :  20 clips |   3.8 min
Data ready ✅


In [3]:
import glob, torch
from huggingface_hub import hf_hub_download
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

V1_DIR = Path(glob.glob("/kaggle/input/**/najdi_xtts_final", recursive=True)[0])
V2_DIR = Path(glob.glob("/kaggle/input/**/najdi_xtts_final_v2", recursive=True)[0])
print("v1:", V1_DIR)
print("v2:", V2_DIR)

BASE_DIR = Path("/kaggle/working/xtts_base")
for f in ["model.pth", "config.json", "vocab.json"]:
    hf_hub_download("coqui/XTTS-v2", f, local_dir=BASE_DIR)

def load_xtts(folder):
    config = XttsConfig()
    config.load_json(str(folder / "config.json"))
    m = Xtts(config)
    m.load_checkpoint(config, checkpoint_path=str(folder / "model.pth"),
                      vocab_path=str(folder / "vocab.json"), use_deepspeed=False)
    m.cuda()
    return m

models = {"base": load_xtts(BASE_DIR), "v1": load_xtts(V1_DIR), "v2": load_xtts(V2_DIR)}
print("All 3 models loaded ✅")

v1: /kaggle/input/notebooks/alaaldeais/tts-fine-tuning/najdi_xtts_final
v2: /kaggle/input/notebooks/alaaldeais/tts-fine-tuning-v2/najdi_xtts_final_v2


model.pth: reconstructing file:   0%|          |  0.00B / 1.87GB            

model.pth: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/4.37k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/361k [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


All 3 models loaded ✅


In [4]:
import random
from IPython.display import Audio, display, Markdown

refs = (meta[(meta["split"] == "train") & meta["dur"].between(7, 12)]
        .groupby("session").head(1).sample(3, random_state=42))
REF_WAVS = [str(OUT_DIR / p) for p in refs["audio_file"]]

test = meta[meta["split"] == "test"].head(5)
sentences = [(t, str(OUT_DIR / a)) for t, a in zip(test["text"], test["audio_file"])]
sentences += [(s, None) for s in [
    "هلا والله، وش أخبارك؟ صار لي زمان ما شفتك.",
    "أبشر، بكرة الصبح أمر عليك ونروح سوا.",
    "تكفى لا تنسى تجيب الأغراض معك وأنت جاي.",
    "لو سمحت، أبي أحجز موعد باسم فهد سلطان الدوسري يوم الأحد.",
    "تكفى أكد على طلب نورة عبدالله القحطاني قبل الساعة خمس.",
]]

EVAL_DIR = Path("/kaggle/working/eval_samples")
EVAL_DIR.mkdir(exist_ok=True)
latents = {n: m.get_conditioning_latents(audio_path=REF_WAVS) for n, m in models.items()}

@torch.inference_mode()
def synth(name, text, seed=42):
    torch.manual_seed(seed)
    gpt_lat, spk = latents[name]
    return np.array(models[name].inference(text, "ar", gpt_lat, spk)["wav"])

rng, key = random.Random(1), []
for i, (text, real) in enumerate(sentences, 1):
    display(Markdown(f"### Sentence {i}\n{text}"))
    if real:
        print("Your real recording:"); display(Audio(real))
    order = list(models.keys()); rng.shuffle(order)
    for label, name in zip(["A", "B", "C"], order):
        wav = synth(name, text)
        sf.write(EVAL_DIR / f"s{i}_{name}.wav", wav, 24000)
        print(f"Version {label}:"); display(Audio(wav, rate=24000))
        key.append({"sentence": i, "label": label, "model": name, "text": text})

pd.DataFrame(key).to_csv(EVAL_DIR / "blind_key.csv", index=False)
print("\nDone — score before revealing!")

### Sentence 1
ممكن ترسل لي رقم المتابعة، لأن التسجيل تم باسم محسن سطام المطيري؟

Your real recording:


Version A:


Version B:


Version C:


### Sentence 2
الموظف أكد أن طلب محسن سطام المطيري انتقل للقسم المختص.

... (36 outputs left out)

Version A:


Version B:


Version C:



Done — score before revealing!


In [5]:
print(pd.read_csv(EVAL_DIR / "blind_key.csv").pivot(index="sentence", columns="label", values="model"))

label        A     B     C
sentence                  
1           v1    v2  base
2           v2  base    v1
3         base    v2    v1
4         base    v2    v1
5           v1    v2  base
6           v2  base    v1
7         base    v2    v1
8           v1  base    v2
9         base    v1    v2
10          v2  base    v1


In [6]:
import librosa

def analyze(path, top_db=35):
    y, sr = librosa.load(path, sr=None)
    iv = librosa.effects.split(y, top_db=top_db)
    gaps = [(iv[i + 1][0] - iv[i][1]) / sr for i in range(len(iv) - 1)]
    return len(y) / sr, max(gaps, default=0.0)

rows = []
for _, r in pd.read_csv(EVAL_DIR / "blind_key.csv").iterrows():
    _, max_gap = analyze(EVAL_DIR / f"s{r['sentence']}_{r['model']}.wav")
    rows.append({"sentence": r["sentence"], "model": r["model"], "longest_pause": round(max_gap, 2)})
res = pd.DataFrame(rows).pivot(index="sentence", columns="model", values="longest_pause")
print(res[["base", "v1", "v2"]])
print("\nAverage longest pause:")
print(res[["base", "v1", "v2"]].mean().round(2))
print("\nSentences with a pause > 0.8 s:")
print((res[["base", "v1", "v2"]] > 0.8).sum())

model     base    v1    v2
sentence                  
1         1.32  0.53  0.32
2         1.28  0.51  0.62
3         1.54  0.55  0.30
4         1.09  0.45  0.75
5         3.41  1.19  0.47
6         1.22  0.92  0.64
7         1.13  0.60  0.47
8         0.64  1.17  0.90
9         1.32  0.75  1.47
10        0.58  1.00  0.53

Average longest pause:
model
base    1.35
v1      0.77
v2      0.65
dtype: float64

Sentences with a pause > 0.8 s:
model
base    8
v1      4
v2      2
dtype: int64
